# Analisis Exploratorio de Datos (EDA) - Video Games Sales

**Fase 1: Comprension de los datos**

Dataset: `Video_Games_Sales_clean.csv`, la version ya limpiada por `app.py` a partir del CSV original
(`Video_Games_Sales_as_at_22_Dec_2016.csv`). La limpieza aplicada incluye: nombres de columna en snake_case,
eliminacion de filas sin identificacion (`name`/`genre`), conversion de `user_score` de texto ("tbd") a
numero, `year_of_release` como entero, y correccion de valores fuera de rango (ventas negativas, scores o
anios invalidos).

Este notebook cubre:
1. Identificacion y clasificacion de variables (numericas / categoricas).
2. Panorama general del dataset y valores faltantes.
3. Estadisticas descriptivas.
4. Visualizaciones y deteccion de patrones.
5. Hallazgos preliminares.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = "Video_Games_Sales_clean.csv"

df = pd.read_csv(DATA_PATH)

# pandas no preserva el tipo Int64 (entero con nulos) al pasar por CSV: una columna
# con nulos se re-infiere como float64. La volvemos a castear a Int64 tal como quedo en app.py.
df["year_of_release"] = df["year_of_release"].astype("Int64")

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.head()

Filas: 16717
Columnas: 16


,critic_count,critic_score,developer,eu_sales,genre,global_sales,jp_sales,na_sales,name,other_sales,platform,publisher,rating,user_count,user_score,year_of_release
0,51.0,76.0,Nintendo,28.96,Sports,82.53,3.77,41.36,Wii Sports,8.45,Wii,Nintendo,E,322.0,8.0,2006
1,NaN,NaN,NaN,3.58,Platform,40.24,6.81,29.08,Super Mario Bros.,0.77,NES,Nintendo,NaN,NaN,NaN,1985
2,73.0,82.0,Nintendo,12.76,Racing,35.52,3.79,15.68,Mario Kart Wii,3.29,Wii,Nintendo,E,709.0,8.3,2008
3,73.0,80.0,Nintendo,10.93,Sports,32.77,3.28,15.61,Wii Sports Resort,2.95,Wii,Nintendo,E,192.0,8.0,2009
4,NaN,NaN,NaN,8.89,Role-Playing,31.37,10.22,11.27,Pokemon Red/Pokemon Blue,1.00,GB,Nintendo,NaN,NaN,NaN,1996


## 1. Identificacion y clasificacion de variables

Antes de analizar nada, separamos las columnas segun el tipo de dato que pandas infirio al leer el CSV.


In [2]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

print(f"Variables numericas ({len(numeric_cols)}): {numeric_cols}")
print(f"Variables categoricas/texto ({len(categorical_cols)}): {categorical_cols}")

Variables numericas (10): ['critic_count', 'critic_score', 'eu_sales', 'global_sales', 'jp_sales', 'na_sales', 'other_sales', 'user_count', 'user_score', 'year_of_release']
Variables categoricas/texto (6): ['developer', 'genre', 'name', 'platform', 'publisher', 'rating']


**Variables numericas (10):** `year_of_release`, `na_sales`, `eu_sales`, `jp_sales`, `other_sales`,
`global_sales`, `critic_score`, `critic_count`, `user_score`, `user_count`. Representan magnitudes (anios,
millones de copias vendidas, cantidad de resenas, puntajes).

**Variables categoricas/texto (6):** `name`, `platform`, `genre`, `publisher`, `developer`, `rating`.
Identifican o clasifican cada juego (texto libre o categorias con pocos valores posibles).

A diferencia del CSV original (donde `user_score` quedaba mal clasificada como texto por el valor
centinela `"tbd"` de Metacritic), aca ya llega correctamente como numerica: esa conversion se hizo una sola
vez en `app.py` en vez de repetirla en cada notebook que consuma el dataset.


## 2. Panorama general del dataset

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16717 entries, 0 to 16716
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   critic_count     8137 non-null   float64
 1   critic_score     8137 non-null   float64
 2   developer        10096 non-null  str    
 3   eu_sales         16717 non-null  float64
 4   genre            16717 non-null  str    
 5   global_sales     16717 non-null  float64
 6   jp_sales         16717 non-null  float64
 7   na_sales         16717 non-null  float64
 8   name             16717 non-null  str    
 9   other_sales      16717 non-null  float64
 10  platform         16717 non-null  str    
 11  publisher        16663 non-null  str    
 12  rating           9950 non-null   str    
 13  user_count       7590 non-null   float64
 14  user_score       7590 non-null   float64
 15  year_of_release  16444 non-null  Int64  
dtypes: Int64(1), float64(9), str(6)
memory usage: 2.1 MB


In [4]:
print("Filas duplicadas:", df.duplicated().sum())
print("Rango de anios:", int(df["year_of_release"].min()), "-", int(df["year_of_release"].max()))
print("Plataformas distintas:", df["platform"].nunique())
print("Generos distintos:", df["genre"].nunique())
print("Publishers distintos:", df["publisher"].nunique())
print("Developers distintos:", df["developer"].nunique())
print("Titulos distintos (name):", df["name"].nunique(), "sobre", len(df), "filas")

Filas duplicadas: 0
Rango de anios: 1980 - 2016
Plataformas distintas: 31
Generos distintos: 12
Publishers distintos: 580
Developers distintos: 1696
Titulos distintos (name): 11562 sobre 16717 filas


No hay filas duplicadas. `name` tiene menos valores unicos que filas: un mismo titulo puede aparecer
varias veces porque se lanzo en distintas plataformas (ej. "FIFA 16" en PS4, Xbox One, PC, etc.), cada fila
es una combinacion **juego + plataforma**, no un juego unico. El rango de anios ahora termina en 2016 porque
`app.py` invalido (paso a nulo) los 4 registros con anio posterior a la fecha de recoleccion del dataset.
